In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error
from src.custom_fastkan import FastKAN
import pandas as pd

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [2]:
def true_function(x):
    # f(x, y) = exp(0.5(sin(\pi(x_1^2 _ x_2^2)) + sin(\pi(x_3^2 + x_4^2))))
    return torch.exp(0.5 * (torch.sin(torch.pi * (x[:, 0]**2 - x[:, 1]**2)) + torch.sin(torch.pi * (x[:, 2]**2 + x[:, 3]**2))))

torch.manual_seed(42)
num_samples = 10000

X = torch.rand(num_samples, 4) * 2 - 1 
y = true_function(X)

train_size = int(0.7 * num_samples)
val_size = int(0.15 * num_samples)
test_size = num_samples - train_size - val_size

X_train, X_val, X_test = torch.split(X, [train_size, val_size, test_size])
y_train, y_val, y_test = torch.split(y, [train_size, val_size, test_size])

X_train = X_train.to(device)
y_train = y_train.to(device)
X_val = X_val.to(device)
y_val = y_val.to(device)
X_test = X_test.to(device)
y_test = y_test.to(device)

print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")

Train shape: torch.Size([7000, 4]), Val shape: torch.Size([1500, 4]), Test shape: torch.Size([1500, 4])


In [3]:
model = FastKAN([4, 4, 2, 1], grid_min=-1, grid_max=1, num_grids=10, use_base_update=False, use_layernorm=False).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [4]:
print("Training KAN...")
train_losses = []
val_losses = []

for epoch in range(1000):
    optimizer.zero_grad()
    pred = model(X_train).squeeze()
    loss = torch.mean((pred - y_train)**2)
    loss.backward()
    optimizer.step()
    
    if epoch % 50 == 0:
        with torch.no_grad():
            val_pred = model(X_val).squeeze()
            val_loss = torch.mean((val_pred - y_val)**2)
            train_losses.append(loss.item())
            val_losses.append(val_loss.item())
            print(f"Epoch {epoch}, Train MSE: {loss.item():.6f}, Val MSE: {val_loss.item():.6f}")

model.eval()
with torch.no_grad():
    test_pred = model(X_test).squeeze()
    mse = mean_squared_error(y_test.cpu(), test_pred.cpu())
    r2 = r2_score(y_test.cpu(), test_pred.cpu())
    
    print(f"KAN Test MSE: {mse:.6f}")
    print(f"KAN Test R2: {r2:.4f}")

Training KAN...
Epoch 0, Train MSE: 2.144930, Val MSE: 1.955249
Epoch 50, Train MSE: 0.234683, Val MSE: 0.218475
Epoch 100, Train MSE: 0.055177, Val MSE: 0.052102
Epoch 150, Train MSE: 0.019294, Val MSE: 0.018833
Epoch 200, Train MSE: 0.007837, Val MSE: 0.008059
Epoch 250, Train MSE: 0.004292, Val MSE: 0.005019
Epoch 300, Train MSE: 0.002722, Val MSE: 0.002969
Epoch 350, Train MSE: 0.002152, Val MSE: 0.002332
Epoch 400, Train MSE: 0.001874, Val MSE: 0.001994
Epoch 450, Train MSE: 0.002169, Val MSE: 0.002370
Epoch 500, Train MSE: 0.001427, Val MSE: 0.001512
Epoch 550, Train MSE: 0.001242, Val MSE: 0.001331
Epoch 600, Train MSE: 0.001464, Val MSE: 0.001378
Epoch 650, Train MSE: 0.001040, Val MSE: 0.001100
Epoch 700, Train MSE: 0.000946, Val MSE: 0.001071
Epoch 750, Train MSE: 0.000813, Val MSE: 0.000874
Epoch 800, Train MSE: 0.000790, Val MSE: 0.000832
Epoch 850, Train MSE: 0.000804, Val MSE: 0.000889
Epoch 900, Train MSE: 0.000686, Val MSE: 0.000782
Epoch 950, Train MSE: 0.000983, Val M

In [5]:
import torch.nn as nn
torch.manual_seed(42)
class MLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dims=[64, 64], output_dim=1):
        super(MLP, self).__init__()
        layers = []
        curr_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(curr_dim, h_dim))
            layers.append(nn.ReLU())
            curr_dim = h_dim
        layers.append(nn.Linear(curr_dim, output_dim))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

mlp_model = MLP(input_dim=4, hidden_dims=[64, 64, 64, 64], output_dim=1).to(device)
mlp_optimizer = torch.optim.Adam(mlp_model.parameters(), lr=0.01)

print("Training MLP...")
for epoch in range(3000):
    mlp_model.train()
    mlp_optimizer.zero_grad()
    pred = mlp_model(X_train).squeeze()
    loss = torch.mean((pred - y_train)**2)
    loss.backward()
    mlp_optimizer.step()
    
    if epoch % 100 == 0:
        mlp_model.eval()
        with torch.no_grad():
            val_pred = mlp_model(X_val).squeeze()
            val_loss = torch.mean((val_pred - y_val)**2)
            print(f"Epoch {epoch}, Train MSE: {loss.item():.6f}, Val MSE: {val_loss.item():.6f}")

mlp_model.eval()
with torch.no_grad():
    mlp_pred = mlp_model(X_test).squeeze()
    mlp_mse = mean_squared_error(y_test.cpu(), mlp_pred.cpu())
    mlp_r2 = r2_score(y_test.cpu(), mlp_pred.cpu())
    
    print(f"\nMLP Test MSE: {mlp_mse:.6f}")
    print(f"MLP Test R2: {mlp_r2:.4f}")

Training MLP...
Epoch 0, Train MSE: 2.123601, Val MSE: 1.696594
Epoch 100, Train MSE: 0.091200, Val MSE: 0.096062
Epoch 200, Train MSE: 0.031398, Val MSE: 0.034108
Epoch 300, Train MSE: 0.012032, Val MSE: 0.015072
Epoch 400, Train MSE: 0.007366, Val MSE: 0.009173
Epoch 500, Train MSE: 0.005830, Val MSE: 0.008092
Epoch 600, Train MSE: 0.004800, Val MSE: 0.006594
Epoch 700, Train MSE: 0.004427, Val MSE: 0.005728
Epoch 800, Train MSE: 0.003438, Val MSE: 0.004843
Epoch 900, Train MSE: 0.003047, Val MSE: 0.004107
Epoch 1000, Train MSE: 0.003125, Val MSE: 0.004405
Epoch 1100, Train MSE: 0.002958, Val MSE: 0.003592
Epoch 1200, Train MSE: 0.002154, Val MSE: 0.003022
Epoch 1300, Train MSE: 0.006437, Val MSE: 0.005711
Epoch 1400, Train MSE: 0.002437, Val MSE: 0.003025
Epoch 1500, Train MSE: 0.002355, Val MSE: 0.003096
Epoch 1600, Train MSE: 0.002103, Val MSE: 0.002900
Epoch 1700, Train MSE: 0.001419, Val MSE: 0.002094
Epoch 1800, Train MSE: 0.001866, Val MSE: 0.002592
Epoch 1900, Train MSE: 0.00

In [6]:
print("\nAnalyzing individual KAN prediction losses...")
individual_losses = []
predictions = []
with torch.no_grad():
    for i in range(len(X_test)):
        input_seq = X_test[i]
        ground_truth = y_test[i]
        prediction = model(input_seq.unsqueeze(0)).squeeze()
        
        loss = ((prediction - ground_truth) ** 2).item()
        individual_losses.append(loss)
        predictions.append(prediction)

individual_losses = np.array(individual_losses)
predictions = torch.stack(predictions)
sorted_indices = np.argsort(individual_losses)

mean_loss = np.mean(individual_losses)
lowest_indices = sorted_indices[:3] 
highest_indices = sorted_indices[-3:]

mean_distances = np.abs(individual_losses - mean_loss)
mean_sorted_indices = np.argsort(mean_distances)
mean_indices = mean_sorted_indices[:3]

print(f"\nKAN Loss Statistics:")
print(f"Mean Loss: {mean_loss:.6f}")
print(f"Min Loss: {individual_losses[lowest_indices[0]]:.6f}")
print(f"Max Loss: {individual_losses[highest_indices[-1]]:.6f}")


Analyzing individual KAN prediction losses...

KAN Loss Statistics:
Mean Loss: 0.000760
Min Loss: 0.000000
Max Loss: 0.026994


In [7]:
print("\nAnalyzing individual MLP prediction losses...")
mlp_individual_losses = []
mlp_predictions = []

mlp_model.eval()
with torch.no_grad():
    for i in range(len(X_test)):
        input_seq = X_test[i]
        ground_truth = y_test[i]
        prediction = mlp_model(input_seq.unsqueeze(0)).squeeze()
        
        loss = ((prediction - ground_truth) ** 2).item()
        mlp_individual_losses.append(loss)
        mlp_predictions.append(prediction)

mlp_individual_losses = np.array(mlp_individual_losses)
mlp_predictions = torch.stack(mlp_predictions)
mlp_sorted_indices = np.argsort(mlp_individual_losses)

mlp_mean_loss = np.mean(mlp_individual_losses)
mlp_lowest_indices = mlp_sorted_indices[:3] 
mlp_highest_indices = mlp_sorted_indices[-3:]

mlp_mean_distances = np.abs(mlp_individual_losses - mlp_mean_loss)
mlp_mean_sorted_indices = np.argsort(mlp_mean_distances)
mlp_mean_indices = mlp_mean_sorted_indices[:3]

print(f"\nMLP Loss Statistics:")
print(f"Mean Loss: {mlp_mean_loss:.6f}")
print(f"Min Loss: {mlp_individual_losses[mlp_lowest_indices[0]]:.6f}")
print(f"Max Loss: {mlp_individual_losses[mlp_highest_indices[-1]]:.6f}")


Analyzing individual MLP prediction losses...

MLP Loss Statistics:
Mean Loss: 0.001484
Min Loss: 0.000000
Max Loss: 0.039584


In [8]:
table_data = []
categories = [("Lowest", lowest_indices), ("Highest", highest_indices), ("Mean", mean_indices)]

for label, indices in categories:
    for idx in indices:
        table_data.append({
            "Category": label,
            "Index": idx,
            "Input (x_1, x_2, x_3, x_4)": f"({X_test[idx][0].item():.4f}, {X_test[idx][1].item():.4f}, {X_test[idx][2].item():.4f}, {X_test[idx][3].item():.4f})",
            "True Value": y_test[idx].item(),
            "Predicted": predictions[idx].item(),
            "Loss": individual_losses[idx]
        })

df_kan_analysis = pd.DataFrame(table_data)
df_kan_analysis


,Category,Index,"Input (x_1, x_2, x_3, x_4)",True Value,Predicted,Loss
0,Lowest,1407,"(0.0283, -0.6534, -0.9509, -0.5406)",0.460286,0.460370,7.068160e-09
1,Lowest,291,"(-0.9241, -0.2111, -0.0930, 0.1864)",1.418825,1.418731,8.868995e-09
2,Lowest,621,"(-0.4211, 0.8975, 0.6171, -0.6360)",0.862625,0.862472,2.335588e-08
3,Highest,875,"(0.9128, -0.9980, 0.3843, 0.5643)",1.286974,1.420175,1.774257e-02
4,Highest,1288,"(0.9162, -0.9930, -0.7950, 0.2161)",1.222433,1.386151,2.680354e-02
5,Highest,714,"(0.9732, 0.9987, -0.8596, -0.4925)",0.951675,1.115973,2.699407e-02
6,Mean,1418,"(0.5736, -0.0573, -0.4356, -0.6407)",2.465475,2.437889,7.609996e-04
7,Mean,382,"(0.3695, 0.3979, 0.4247, -0.8937)",0.998756,0.971167,7.611247e-04
8,Mean,29,"(-0.4822, 0.4525, 0.6348, -0.9802)",0.662780,0.690442,7.651884e-04


In [9]:
table_data_mlp = []
categories = [("Lowest", mlp_lowest_indices), ("Highest", mlp_highest_indices), ("Mean", mlp_mean_indices)]

for label, indices in categories:
    for idx in indices:
        table_data_mlp.append({
            "Category": label,
            "Index": idx,
            "Input (x_1, x_2, x_3, x_4)": f"({X_test[idx][0].item():.4f}, {X_test[idx][1].item():.4f}, {X_test[idx][2].item():.4f}, {X_test[idx][3].item():.4f})",
            "True Value": y_test[idx].item(),
            "Predicted": mlp_predictions[idx].item(),
            "Loss": mlp_individual_losses[idx]
        })

df_mlp_analysis = pd.DataFrame(table_data_mlp)
df_mlp_analysis


,Category,Index,"Input (x_1, x_2, x_3, x_4)",True Value,Predicted,Loss
0,Lowest,9,"(-0.7327, 0.5764, 0.8102, -0.7603)",0.964494,0.964501,5.551115e-11
1,Lowest,937,"(-0.1727, -0.2938, 0.6635, -0.2212)",1.509026,1.509017,9.094947e-11
2,Lowest,1479,"(-0.8782, -0.4648, 0.0896, -0.1678)",1.731862,1.731835,7.517542e-10
3,Highest,1432,"(-0.9776, 0.3050, -0.9958, 0.9909)",1.182119,1.018872,2.664966e-02
4,Highest,1181,"(0.9834, -0.0315, 0.6910, -0.1553)",1.738845,1.903750,2.719345e-02
5,Highest,357,"(0.9935, -0.2855, 0.8736, 0.0208)",1.623433,1.822392,3.958430e-02
6,Mean,889,"(0.9279, 0.7146, -0.0618, 0.2578)",1.742450,1.781041,1.489212e-03
7,Mean,794,"(0.0379, -0.0595, -0.2686, -0.0774)",1.125436,1.163891,1.478797e-03
8,Mean,681,"(0.8196, 0.3322, 0.8564, 0.6402)",1.314075,1.275651,1.476377e-03


In [10]:
mlp_save_path = "model_pkls/functionexp4_mlp_model.pkl"
torch.save({
    'model_state_dict': mlp_model.state_dict(),
    'config': {
        'input_dim': 4,
        'hidden_dims': [64, 64, 64, 64],
        'output_dim': 1
    }
}, mlp_save_path)
print(f"MLP Model saved to {mlp_save_path}")

MLP Model saved to model_pkls/functionexp4_mlp_model.pkl


In [11]:
kan_save_path = "model_pkls/functionexp4_kan_model.pkl"
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'layers_hidden': [4, 4, 2, 1],
        'grid_min': -1,
        'grid_max': 1,
        'num_grids': 10,
        'use_base_update': False,
        'use_layernorm': False,
    }
}, kan_save_path)
print(f"KAN Model saved to {kan_save_path}")

KAN Model saved to model_pkls/functionexp4_kan_model.pkl
